In [16]:
# =============================================================================
# OPTIMIZED MANUFACTURER RISK ASSESSMENT SYSTEM - FIXED VERSION
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import beta, triang
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [17]:
# =============================================================================
# SECTION 1: ENHANCED RISK COMPONENTS CONFIGURATION
# =============================================================================

@dataclass
class RiskComponents:
    """
    Enhanced configuration class for risk assessment components.
    Optimized for better risk differentiation.
    """

    # Probability weights - optimized for better differentiation
    PROB_WEIGHTS = {
        'acquisition_risk': 0.35,
        'geographic_risk': 0.25,
        'business_stability': 0.25,
        'transparency': 0.10,
        'manufacturing_diversity': 0.05
    }

    # Impact weights
    IMPACT_WEIGHTS = {
        'supply_chain_criticality': 0.35,
        'business_interruption': 0.25,
        'regulatory_compliance': 0.20,
        'market_position': 0.20
    }

    # Exposure weights
    EXPOSURE_WEIGHTS = {
        'market_footprint': 0.40,
        'geographic_exposure': 0.30,
        'supply_chain_visibility': 0.30
    }

    # OPTIMIZED Risk scoring thresholds
    RISK_THRESHOLDS = {
        'critical': 0.50,
        'high': 0.35,
        'medium': 0.20,
        'low': 0.10
    }

    # Country risk scores
    COUNTRY_RISK_SCORES = {
        'USA': 0.35,
        'Japan': 0.25,
        'Taiwan': 0.60,
        'United Kingdom': 0.30,
        'Netherlands': 0.25,
        'Switzerland': 0.25,
        'Singapore': 0.30,
        'Germany': 0.25,
        'China': 0.70,
        'Unknown': 0.50
    }

    # Acquisition age risk factors
    ACQUISITION_AGE_RISK = {
        'recent': 1.3,
        'medium': 1.15,
        'old': 1.0
    }


In [18]:
# =============================================================================
# SECTION 2: ENHANCED VALIDATED DATA LOADER
# =============================================================================

class ValidatedDataLoader:
    def __init__(self, data_path: str = None, dataframe: pd.DataFrame = None):
        self.data_path = data_path
        self.df = dataframe
        self.validation_report = {}

    def load_data(self) -> pd.DataFrame:
        try:
            if self.df is not None:
                df = self.df.copy()
            elif self.data_path:
                df = pd.read_excel(self.data_path, sheet_name='Sheet2')
            else:
                raise ValueError("No data source provided")

            required_cols = ['id', 'mfr', 'country', 'headquarters', 'status', 'website']
            missing_cols = [col for col in required_cols if col not in df.columns]

            if missing_cols:
                raise ValueError(f"Missing required columns: {missing_cols}")

            self.validation_report = {
                'total_manufacturers': len(df),
                'missing_values': df.isnull().sum().to_dict(),
                'active_manufacturers': len(df[df['status'].str.contains('Active', case=False, na=False)]),
                'acquired_manufacturers': len(df[~df['status'].str.contains('Active', case=False, na=False)])
            }

            df = self._create_enhanced_features(df)
            return df

        except Exception as e:
            print(f"Error loading data: {e}")
            raise

    def _create_enhanced_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Parse headquarters
        hq_split = df['headquarters'].str.split(',', n=1, expand=True)
        df['hq_city'] = hq_split[0].str.strip()
        df['hq_state_region'] = hq_split[1].str.strip() if hq_split.shape[1] > 1 else 'Unknown'

        # Enhanced status features
        df['is_acquired'] = ~df['status'].str.contains('Active', case=False, na=False)
        df['acquisition_year'] = df['status'].str.extract(r'(\d{4})').fillna(0).astype(int)

        current_year = 2026
        df['acquisition_age'] = df['acquisition_year'].apply(
            lambda x: current_year - x if x > 0 else 0
        )

        # Business stability score
        df['business_stability'] = df.apply(
            lambda row: 0.2 if row['is_acquired'] and row['acquisition_age'] < 3
            else 0.4 if row['is_acquired'] and row['acquisition_age'] < 7
            else 0.6 if row['is_acquired']
            else 0.9,
            axis=1
        )

        df['country_risk'] = df['country'].map(RiskComponents.COUNTRY_RISK_SCORES).fillna(0.5)

        df['website_quality'] = df['website'].apply(
            lambda x: 0.9 if pd.notna(x) and ('https' in str(x) or 'http' in str(x))
            else 0.5 if pd.notna(x)
            else 0.2
        )

        df['supply_criticality'] = df.apply(
            lambda row: 0.8 if row['country'] in ['USA', 'Japan', 'Germany', 'Taiwan']
            else 0.6 if row['country'] in ['Netherlands', 'Switzerland', 'Singapore']
            else 0.5,
            axis=1
        )

        df['market_footprint'] = df['country'].apply(
            lambda x: 0.8 if x in ['USA']
            else 0.7 if x in ['Japan', 'Germany', 'Netherlands', 'Switzerland']
            else 0.6 if x in ['Taiwan', 'Singapore', 'United Kingdom']
            else 0.5
        )

        df['compliance_risk'] = df['country'].apply(
            lambda x: 0.3 if x in ['USA', 'Germany', 'Japan']
            else 0.5 if x in ['United Kingdom', 'Netherlands', 'Switzerland']
            else 0.7
        )

        country_counts = df['country'].value_counts()
        df['country_concentration'] = df['country'].map(
            lambda x: country_counts.get(x, 0) / len(df)
        )

        return df

    def get_validation_report(self) -> Dict:
        return self.validation_report


In [19]:
# =============================================================================
# SECTION 3: ENHANCED TIME SERIES FEATURE ENGINEER
# =============================================================================

class TimeSeriesFeatureEngineer:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def engineer_features(self) -> pd.DataFrame:
        df = self.df.copy()

        # Acquisition risk
        df['acquisition_risk'] = df.apply(
            lambda row: 0.9 if row['is_acquired'] and row['acquisition_age'] < 3
            else 0.7 if row['is_acquired'] and row['acquisition_age'] < 7
            else 0.5 if row['is_acquired']
            else 0.2,
            axis=1
        )

        # Geographic risk with concentration penalty
        df['geographic_risk_score'] = df.apply(
            lambda row: min(row['country_risk'] * (1 + row['country_concentration']), 1.0),
            axis=1
        )

        # Supply chain disruption risk
        df['disruption_risk'] = df.apply(
            lambda row: 0.7 if row['is_acquired'] and row['acquisition_age'] < 3
            else 0.5 if row['is_acquired']
            else 0.3 if row['country_risk'] > 0.5
            else 0.2,
            axis=1
        )

        df['diversity_score'] = df['country'].apply(
            lambda x: 0.7 if x in ['USA']
            else 0.5 if x in ['Japan', 'Germany']
            else 0.3
        )

        df['regulatory_exposure'] = df['country'].apply(
            lambda x: 0.4 if x in ['USA', 'Germany', 'Japan']
            else 0.6 if x in ['Taiwan', 'China']
            else 0.5
        )

        df['industry_stability'] = df.apply(
            lambda row: 0.7 if not row['is_acquired'] else 0.4,
            axis=1
        )

        df['supply_visibility'] = df['website_quality']

        df['manufacturing_concentration'] = df.apply(
            lambda row: 0.6 if row['country'] in ['USA', 'Japan', 'Germany']
            else 0.8 if row['country'] in ['Taiwan', 'China']
            else 0.5,
            axis=1
        )

        # Data completeness score
        df['data_completeness'] = df.apply(
            lambda row: sum([
                1 if pd.notna(row.get('country')) else 0,
                1 if pd.notna(row.get('headquarters')) else 0,
                1 if pd.notna(row.get('website')) else 0,
                1 if row.get('acquisition_year', 0) > 0 else 0
            ]) / 4,
            axis=1
        )

        return df


In [21]:
# =============================================================================
# SECTION 4: BAYESIAN RISK ESTIMATOR
# =============================================================================

class BayesianRiskEstimator:
    def __init__(self, confidence_level: float = 0.95):
        self.confidence_level = confidence_level
        self.posterior_samples = None

    def estimate_probability(self, observed_successes: int, observed_failures: int,
                            prior_a: float = 2.0, prior_b: float = 2.0) -> Dict:
        posterior_a = prior_a + observed_successes
        posterior_b = prior_b + observed_failures

        self.posterior_samples = beta.rvs(posterior_a, posterior_b, size=10000)

        return {
            'mean_probability': np.mean(self.posterior_samples),
            'std_deviation': np.std(self.posterior_samples),
            'credible_interval_lower': np.percentile(self.posterior_samples, 2.5),
            'credible_interval_upper': np.percentile(self.posterior_samples, 97.5),
            'posterior_parameters': {'a': posterior_a, 'b': posterior_b}
        }

In [22]:
# =============================================================================
# SECTION 5: MONTE CARLO RISK SIMULATOR
# =============================================================================

class MonteCarloRiskSimulator:
    def __init__(self, n_simulations: int = 10000):
        self.n_simulations = n_simulations
        self.results = None

    def simulate_risk(self, probability: float, impact: float, exposure: float,
                     uncertainty: float = 0.15) -> Dict:
        prob_dist = triang(0.5, max(0, probability - uncertainty/2), min(1, probability + uncertainty/2))
        impact_dist = triang(0.5, max(0, impact - uncertainty/2), min(1, impact + uncertainty/2))
        exposure_dist = triang(0.5, max(0, exposure - uncertainty/2), min(1, exposure + uncertainty/2))

        prob_samples = np.clip(prob_dist.rvs(self.n_simulations), 0, 1)
        impact_samples = np.clip(impact_dist.rvs(self.n_simulations), 0, 1)
        exposure_samples = np.clip(exposure_dist.rvs(self.n_simulations), 0, 1)

        risk_scores = prob_samples * impact_samples * exposure_samples

        self.results = {
            'risk_scores': risk_scores,
            'mean_risk': np.mean(risk_scores),
            'std_risk': np.std(risk_scores),
            'percentile_5': np.percentile(risk_scores, 5),
            'percentile_50': np.percentile(risk_scores, 50),
            'percentile_95': np.percentile(risk_scores, 95)
        }
        return self.results

    def get_scenario_analysis(self) -> Dict:
        if self.results is None:
            return {}
        return {
            'best_case': self.results['percentile_5'],
            'expected': self.results['mean_risk'],
            'worst_case': self.results['percentile_95'],
            'volatility': self.results['std_risk']
        }

In [23]:
# =============================================================================
# SECTION 6: OPTIMIZED PROPER RISK SCORER
# =============================================================================

class ProperRiskScorer:
    def __init__(self):
        self.components = RiskComponents()

    def calculate_probability(self, row: pd.Series) -> float:
        weights = self.components.PROB_WEIGHTS

        acquisition = row.get('acquisition_risk', 0.3)
        geographic = row.get('geographic_risk_score', 0.3)
        stability = 1 - row.get('business_stability', 0.7)
        transparency = 1 - row.get('supply_visibility', 0.5)
        diversity = 1 - row.get('diversity_score', 0.5)

        prob = (
            weights['acquisition_risk'] * acquisition +
            weights['geographic_risk'] * geographic +
            weights['business_stability'] * stability +
            weights['transparency'] * transparency +
            weights['manufacturing_diversity'] * diversity
        )

        prob = 0.1 + prob * 0.8
        return np.clip(prob, 0.1, 0.95)

    def calculate_impact(self, row: pd.Series) -> float:
        weights = self.components.IMPACT_WEIGHTS

        criticality = row.get('supply_criticality', 0.5)
        interruption = row.get('disruption_risk', 0.3)
        compliance = row.get('regulatory_exposure', 0.4)
        market = row.get('market_footprint', 0.5)

        impact = (
            weights['supply_chain_criticality'] * criticality +
            weights['business_interruption'] * interruption +
            weights['regulatory_compliance'] * compliance +
            weights['market_position'] * market
        )

        impact = 0.1 + impact * 0.8
        return np.clip(impact, 0.1, 0.95)

    def calculate_exposure(self, row: pd.Series) -> float:
        weights = self.components.EXPOSURE_WEIGHTS

        footprint = row.get('market_footprint', 0.5)
        geo_exposure = row.get('geographic_risk_score', 0.3)
        visibility = 1 - row.get('supply_visibility', 0.5)

        exposure = (
            weights['market_footprint'] * footprint +
            weights['geographic_exposure'] * geo_exposure +
            weights['supply_chain_visibility'] * visibility
        )

        exposure = 0.1 + exposure * 0.8
        return np.clip(exposure, 0.1, 0.95)

    def calculate_confidence_score(self, row: pd.Series) -> float:
        base_confidence = 0.85
        data_completeness = row.get('data_completeness', 0.7)
        website_bonus = 0.05 if row.get('website_quality', 0) > 0.5 else 0

        confidence = base_confidence * (0.7 + 0.3 * data_completeness) + website_bonus
        return np.clip(confidence, 0.65, 0.98)

    def get_risk_level(self, risk_score: float) -> str:
        thresholds = self.components.RISK_THRESHOLDS

        if risk_score >= thresholds['critical']:
            return 'Critical'
        elif risk_score >= thresholds['high']:
            return 'High'
        elif risk_score >= thresholds['medium']:
            return 'Medium'
        else:
            return 'Low'

In [24]:
# =============================================================================
# SECTION 7: OPTIMIZED SCALABLE DATA PROCESSOR
# =============================================================================

class ScalableDataProcessor:
    def __init__(self):
        self.scorer = ProperRiskScorer()
        self.bayesian = BayesianRiskEstimator()
        self.monte_carlo = MonteCarloRiskSimulator()

    def process_manufacturer(self, row: pd.Series) -> Dict:
        probability = self.scorer.calculate_probability(row)
        impact = self.scorer.calculate_impact(row)
        exposure = self.scorer.calculate_exposure(row)

        risk_score = probability * impact * exposure

        # Apply acquisition multiplier
        if row.get('is_acquired', False):
            age = row.get('acquisition_age', 0)
            if age <= 3:
                multiplier = RiskComponents.ACQUISITION_AGE_RISK['recent']
            elif age <= 7:
                multiplier = RiskComponents.ACQUISITION_AGE_RISK['medium']
            else:
                multiplier = RiskComponents.ACQUISITION_AGE_RISK['old']
            risk_score *= multiplier

        risk_score = np.clip(risk_score, 0.05, 0.95)
        confidence = self.scorer.calculate_confidence_score(row)
        risk_level = self.scorer.get_risk_level(risk_score)

        bayesian_result = self.bayesian.estimate_probability(
            observed_successes=int(risk_score * 15),
            observed_failures=int((1 - risk_score) * 15)
        )

        simulation_result = self.monte_carlo.simulate_risk(
            probability=probability,
            impact=impact,
            exposure=exposure,
            uncertainty=0.15
        )

        scenarios = self.monte_carlo.get_scenario_analysis()

        return {
            'manufacturer': row.get('mfr', 'Unknown'),
            'probability': probability,
            'impact': impact,
            'exposure': exposure,
            'risk_score': risk_score,
            'risk_level': risk_level,
            'confidence_score': confidence,
            'is_acquired': row.get('is_acquired', False),
            'acquisition_age': row.get('acquisition_age', 0),
            'acquisition_year': row.get('acquisition_year', 0),
            'country': row.get('country', 'Unknown'),
            'status': row.get('status', 'Unknown'),
            'data_completeness': row.get('data_completeness', 0),
            'bayesian_estimate': bayesian_result,
            'simulation': simulation_result,
            'scenarios': scenarios
        }

    def process_batch(self, df: pd.DataFrame) -> pd.DataFrame:
        results = []
        for _, row in df.iterrows():
            result = self.process_manufacturer(row)
            results.append(result)
        return pd.DataFrame(results)

In [25]:
# =============================================================================
# SECTION 8: OPTIMIZED COMPREHENSIVE RISK ASSESSMENT SYSTEM
# =============================================================================

class ComprehensiveRiskAssessmentSystem:
    def __init__(self, data_path: str = None, dataframe: pd.DataFrame = None):
        self.data_loader = ValidatedDataLoader(data_path, dataframe)
        self.processor = ScalableDataProcessor()
        self.results = None
        self.risk_summary = None

    def run_assessment(self) -> Dict:
        print("=" * 80)
        print("OPTIMIZED MANUFACTURER RISK ASSESSMENT SYSTEM - FINAL VERSION")
        print("=" * 80)

        print("\n[1] Loading and validating data...")
        df = self.data_loader.load_data()
        validation_report = self.data_loader.get_validation_report()
        print(f"    ✓ Loaded {validation_report['total_manufacturers']} manufacturers")
        print(f"    ✓ Active: {validation_report['active_manufacturers']}")
        print(f"    ✓ Acquired: {validation_report['acquired_manufacturers']}")

        print("\n[2] Engineering features...")
        engineer = TimeSeriesFeatureEngineer(df)
        df_engineered = engineer.engineer_features()
        print(f"    ✓ Created {len(df_engineered.columns)} features")

        print("\n[3] Processing risk assessment...")
        self.results = self.processor.process_batch(df_engineered)
        print(f"    ✓ Assessed {len(self.results)} manufacturers")

        self.risk_summary = self._generate_summary()
        print(f"\n[4] Risk Summary:")
        print(f"    ✓ Average Risk Score: {self.risk_summary['avg_risk']:.3f}")
        print(f"    ✓ Max Risk Score: {self.risk_summary['max_risk']:.3f}")
        print(f"    ✓ Min Risk Score: {self.risk_summary['min_risk']:.3f}")
        print(f"    ✓ High Risk Count: {self.risk_summary['high_risk_count']}")
        print(f"    ✓ Critical Risk Count: {self.risk_summary['critical_risk_count']}")

        risk_dist = self.risk_summary['risk_distribution']
        print(f"    ✓ Risk Level Distribution:")
        for level in ['Critical', 'High', 'Medium', 'Low']:
            count = risk_dist.get(level, 0)
            pct = count / self.risk_summary['total_assessed'] * 100 if self.risk_summary['total_assessed'] > 0 else 0
            print(f"        - {level}: {count} ({pct:.1f}%)")

        return self.risk_summary

    def _generate_summary(self) -> Dict:
        if self.results is None:
            return {}

        risk_levels = self.results['risk_level'].value_counts()

        return {
            'total_assessed': len(self.results),
            'avg_risk': self.results['risk_score'].mean(),
            'max_risk': self.results['risk_score'].max(),
            'min_risk': self.results['risk_score'].min(),
            'std_risk': self.results['risk_score'].std(),
            'risk_distribution': risk_levels.to_dict(),
            'high_risk_count': risk_levels.get('High', 0) + risk_levels.get('Critical', 0),
            'critical_risk_count': risk_levels.get('Critical', 0),
            'avg_confidence': self.results['confidence_score'].mean(),
            'acquired_count': self.results['is_acquired'].sum(),
            'avg_acquisition_age': self.results[self.results['is_acquired']]['acquisition_age'].mean() if self.results['is_acquired'].any() else 0
        }

    def get_top_risk_manufacturers(self, n: int = 10) -> pd.DataFrame:
        if self.results is None:
            return pd.DataFrame()

        top_risk = self.results.nlargest(n, 'risk_score')[
            ['manufacturer', 'risk_score', 'risk_level', 'probability',
             'impact', 'exposure', 'confidence_score', 'country', 'status',
             'is_acquired', 'acquisition_age', 'acquisition_year']
        ].copy()

        return top_risk

    def visualize_results(self):
        """Create comprehensive optimized visualizations."""
        if self.results is None:
            print("No results to visualize.")
            return

        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=('Risk Score Distribution', 'Risk Level Distribution',
                          'Top 10 High-Risk Manufacturers', 'Risk Component Analysis',
                          'Risk by Status', 'Risk Score Distribution by Status'),
            specs=[[{'type': 'histogram'}, {'type': 'pie'}],
                   [{'type': 'bar'}, {'type': 'scatter'}],
                   [{'type': 'box'}, {'type': 'box'}]]
        )

        # 1. Risk score distribution
        fig.add_trace(
            go.Histogram(x=self.results['risk_score'],
                        nbinsx=20,
                        name='Risk Distribution',
                        marker_color='coral'),
            row=1, col=1
        )

        # 2. Risk level distribution
        risk_counts = self.results['risk_level'].value_counts()
        colors = {'Critical': 'darkred', 'High': 'red', 'Medium': 'orange', 'Low': 'green'}
        fig.add_trace(
            go.Pie(labels=risk_counts.index,
                   values=risk_counts.values,
                   name='Risk Levels',
                   marker=dict(colors=[colors.get(l, 'blue') for l in risk_counts.index])),
            row=1, col=2
        )

        # 3. Top 10 high-risk manufacturers
        top_risk = self.get_top_risk_manufacturers(10)
        fig.add_trace(
            go.Bar(x=top_risk['manufacturer'][:10],
                   y=top_risk['risk_score'][:10],
                   name='Top Risk',
                   marker_color='crimson',
                   text=top_risk['risk_level'][:10],
                   textposition='outside'),
            row=2, col=1
        )

        # 4. Risk component analysis
        fig.add_trace(
            go.Scatter(x=self.results['probability'],
                      y=self.results['impact'],
                      mode='markers',
                      marker=dict(size=self.results['exposure']*80 + 10,
                                 color=self.results['risk_score'],
                                 colorscale='Reds',
                                 showscale=True,
                                 colorbar=dict(title='Risk Score')),
                      text=[f"{m}<br>Risk: {r:.3f}" for m, r in zip(self.results['manufacturer'], self.results['risk_score'])],
                      name='Risk Components'),
            row=2, col=2
        )

        # 5. Risk by status (box plot)
        status_data = self.results.copy()
        status_data['status_group'] = status_data['is_acquired'].map({True: 'Acquired', False: 'Active'})
        fig.add_trace(
            go.Box(x=status_data['status_group'],
                   y=status_data['risk_score'],
                   name='Risk by Status',
                   marker_color='lightblue',
                   boxmean='sd'),
            row=3, col=1
        )

        # 6. Risk distribution by status
        acquired_risk = self.results[self.results['is_acquired']]['risk_score']
        active_risk = self.results[~self.results['is_acquired']]['risk_score']

        fig.add_trace(
            go.Box(y=active_risk, name='Active', marker_color='green', boxmean='sd'),
            row=3, col=2
        )
        fig.add_trace(
            go.Box(y=acquired_risk, name='Acquired', marker_color='red', boxmean='sd'),
            row=3, col=2
        )

        fig.update_layout(height=900, width=1400,
                         title_text="OPTIMIZED MANUFACTURER RISK ASSESSMENT RESULTS",
                         showlegend=True)
        fig.show()

        self._create_detailed_visualizations()

    def _create_detailed_visualizations(self):
        """Create additional detailed visualizations."""
        # Risk by country
        country_risk = self.results.groupby('country')['risk_score'].agg(['mean', 'count']).sort_values('mean', ascending=False)
        country_risk['percentage'] = country_risk['count'] / len(self.results) * 100

        fig2 = make_subplots(rows=1, cols=2,
                             subplot_titles=('Average Risk by Country', 'Risk-Confidence Relationship'))

        fig2.add_trace(
            go.Bar(x=country_risk.index[:8], y=country_risk['mean'][:8],
                  marker_color='darkblue',
                  text=[f"{c:.3f}" for c in country_risk['mean'][:8]],
                  textposition='outside',
                  name='Country Risk'),
            row=1, col=1
        )

        fig2.add_trace(
            go.Scatter(x=self.results['confidence_score'],
                      y=self.results['risk_score'],
                      mode='markers',
                      marker=dict(size=12,
                                 color=self.results['risk_score'],
                                 colorscale='Viridis',
                                 showscale=True,
                                 colorbar=dict(title='Risk Score')),
                      text=self.results['manufacturer'],
                      name='Risk-Confidence'),
            row=1, col=2
        )

        fig2.update_layout(height=400, width=1000,
                          title_text="Detailed Risk Analysis")
        fig2.show()

    def generate_report(self) -> str:
        if self.results is None:
            return "No results available."

        report = []
        report.append("=" * 80)
        report.append("OPTIMIZED MANUFACTURER RISK ASSESSMENT REPORT")
        report.append("=" * 80)
        report.append(f"\nAssessment Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
        report.append(f"Total Manufacturers Assessed: {len(self.results)}")

        report.append("\n" + "-" * 40)
        report.append("RISK SUMMARY")
        report.append("-" * 40)
        report.append(f"Average Risk Score: {self.risk_summary['avg_risk']:.3f}")
        report.append(f"Maximum Risk Score: {self.risk_summary['max_risk']:.3f}")
        report.append(f"Minimum Risk Score: {self.risk_summary['min_risk']:.3f}")
        report.append(f"Risk Score Std Dev: {self.risk_summary['std_risk']:.3f}")
        report.append(f"Average Confidence Score: {self.risk_summary['avg_confidence']:.3f}")

        report.append("\n" + "-" * 40)
        report.append("RISK LEVEL DISTRIBUTION")
        report.append("-" * 40)
        for level in ['Critical', 'High', 'Medium', 'Low']:
            count = self.risk_summary['risk_distribution'].get(level, 0)
            percentage = count / len(self.results) * 100
            report.append(f"{level}: {count} ({percentage:.1f}%)")

        report.append("\n" + "-" * 40)
        report.append("TOP HIGH-RISK MANUFACTURERS")
        report.append("-" * 40)
        top_risk = self.get_top_risk_manufacturers(10)
        for _, row in top_risk.iterrows():
            report.append(f"\nManufacturer: {row['manufacturer']}")
            report.append(f"  Risk Score: {row['risk_score']:.3f}")
            report.append(f"  Risk Level: {row['risk_level']}")
            report.append(f"  Country: {row['country']}")
            report.append(f"  Status: {row['status']}")
            if row['is_acquired']:
                report.append(f"  ⚠ Acquired ({row['acquisition_age']} years ago)")

        report.append("\n" + "-" * 40)
        report.append("ACTIONABLE RECOMMENDATIONS")
        report.append("-" * 40)
        report.append(self._generate_recommendations())

        return "\n".join(report)

    def _generate_recommendations(self) -> str:
        if self.results is None:
            return ""

        recommendations = []

        # Medium risk manufacturers
        medium_risk = self.results[self.results['risk_level'] == 'Medium']
        if len(medium_risk) > 0:
            recommendations.append("⚠️ MEDIUM RISK MANUFACTURERS - MONITOR CLOSELY:")
            for _, row in medium_risk.iterrows():
                rec = f"  • {row['manufacturer']} (Risk: {row['risk_score']:.3f})"
                if row['is_acquired']:
                    rec += f" - Acquired {row['acquisition_age']} years ago"
                recommendations.append(rec)
            recommendations.append("")

        # Geographic concentration
        country_dist = self.results['country'].value_counts()
        for country, count in country_dist.head(3).items():
            pct = count / len(self.results) * 100
            if pct > 30:
                recommendations.append(f"🌍 GEOGRAPHIC CONCENTRATION: {count} ({pct:.1f}%) manufacturers in {country}")
                recommendations.append(f"  • High geographic concentration risk detected")
                recommendations.append(f"  • Consider diversifying supplier base")
                recommendations.append("")

        # Acquisition risk
        acquired = self.results[self.results['is_acquired']]
        if len(acquired) > 0:
            recommendations.append(f"🔄 ACQUISITION RISK SUMMARY:")
            recommendations.append(f"  • Total Acquired: {len(acquired)} manufacturers")
            recommendations.append(f"  • Average Acquisition Age: {self.risk_summary['avg_acquisition_age']:.1f} years")
            recommendations.append(f"  • Monitor post-acquisition integration")
            recommendations.append("")

        # Data quality
        low_confidence = self.results[self.results['confidence_score'] < 0.75]
        if len(low_confidence) > 0:
            recommendations.append(f"📋 DATA QUALITY CONCERNS:")
            for _, row in low_confidence.iterrows():
                recommendations.append(f"  • {row['manufacturer']} - Low confidence ({row['confidence_score']:.2f})")
            recommendations.append("")

        if not recommendations:
            recommendations.append("✅ All manufacturers are within acceptable risk parameters.")

        return "\n".join(recommendations)

In [10]:
# =============================================================================
# SECTION 9: Main Execution
# =============================================================================

def main():
    """
    Main execution function for Manufacturer Risk Assessment System.
    """
    print("Starting Manufacturer Risk Assessment System...")

    # Initialize the system with the Excel data
    system = ComprehensiveRiskAssessmentSystem(dataframe=pd.read_excel('/content/Manufactor.xlsx', sheet_name='Sheet2'))

    # Run assessment
    summary = system.run_assessment()

    # Generate visualizations
    print("\n[5] Generating visualizations...")
    system.visualize_results()

    # Generate report
    print("\n[6] Generating report...")
    report = system.generate_report()
    print(report)

    # Export results
    print("\n[7] Exporting results...")
    system.results.to_csv('manufacturer_risk_assessment_results.csv', index=False)
    print("    ✓ Results exported to 'manufacturer_risk_assessment_results.csv'")

    # Save report
    with open('manufacturer_risk_assessment_report.txt', 'w') as f:
        f.write(report)
    print("    ✓ Report saved to 'manufacturer_risk_assessment_report.txt'")

    print("\n" + "=" * 80)
    print("RISK ASSESSMENT COMPLETE")
    print("=" * 80)

    return system


In [26]:
# =============================================================================
# SECTION 9: MAIN EXECUTION
# =============================================================================

def main():
    print("Starting Optimized Manufacturer Risk Assessment System...")
    print("=" * 80)

    try:
        system = ComprehensiveRiskAssessmentSystem(
            dataframe=pd.read_excel('/content/Manufactor.xlsx', sheet_name='Sheet2')
        )

        summary = system.run_assessment()

        print("\n[5] Generating visualizations...")
        system.visualize_results()

        print("\n[6] Generating report...")
        report = system.generate_report()
        print(report)

        print("\n[7] Exporting results...")
        system.results.to_csv('manufacturer_risk_assessment_final.csv', index=False)
        print("    ✓ Results exported to 'manufacturer_risk_assessment_final.csv'")

        with open('manufacturer_risk_assessment_report_final.txt', 'w', encoding='utf-8') as f:
            f.write(report)
        print("    ✓ Report saved to 'manufacturer_risk_assessment_report_final.txt'")

        print("\n" + "=" * 80)
        print("OPTIMIZED RISK ASSESSMENT COMPLETE")
        print("=" * 80)

        return system

    except FileNotFoundError:
        print("Error: 'Manufactor.xlsx' not found.")
        return None
    except Exception as e:
        print(f"Error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None

In [27]:
# =============================================================================
# SECTION 10: EXECUTION
# =============================================================================

if __name__ == "__main__":
    system = main()

Starting Optimized Manufacturer Risk Assessment System...
OPTIMIZED MANUFACTURER RISK ASSESSMENT SYSTEM - FINAL VERSION

[1] Loading and validating data...
    ✓ Loaded 34 manufacturers
    ✓ Active: 27
    ✓ Acquired: 7

[2] Engineering features...
    ✓ Created 27 features

[3] Processing risk assessment...
    ✓ Assessed 34 manufacturers

[4] Risk Summary:
    ✓ Average Risk Score: 0.120
    ✓ Max Risk Score: 0.267
    ✓ Min Risk Score: 0.054
    ✓ High Risk Count: 0
    ✓ Critical Risk Count: 0
    ✓ Risk Level Distribution:
        - Critical: 0 (0.0%)
        - High: 0 (0.0%)
        - Medium: 2 (5.9%)
        - Low: 32 (94.1%)

[5] Generating visualizations...



[6] Generating report...
OPTIMIZED MANUFACTURER RISK ASSESSMENT REPORT

Assessment Date: 2026-07-15 06:49
Total Manufacturers Assessed: 34

----------------------------------------
RISK SUMMARY
----------------------------------------
Average Risk Score: 0.120
Maximum Risk Score: 0.267
Minimum Risk Score: 0.054
Risk Score Std Dev: 0.050
Average Confidence Score: 0.800

----------------------------------------
RISK LEVEL DISTRIBUTION
----------------------------------------
Critical: 0 (0.0%)
High: 0 (0.0%)
Medium: 2 (5.9%)
Low: 32 (94.1%)

----------------------------------------
TOP HIGH-RISK MANUFACTURERS
----------------------------------------

Manufacturer: Sanyo
  Risk Score: 0.267
  Risk Level: Medium
  Country: Japan
  Status: Acquired
  ⚠ Acquired (0 years ago)

Manufacturer: Analog Devices/Maxim Integrated
  Risk Score: 0.250
  Risk Level: Medium
  Country: USA
  Status: Acquired by ADI (2021)
  ⚠ Acquired (5 years ago)

Manufacturer: Fairchild Semiconductor
  Risk Score: 0.